In [26]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import os
import mlflow

db_path = os.path.abspath("/workspaces/MLOPS/03-Experiment_tracking/mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")



def read_dataframe(path):
    df=pd.read_parquet(path)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    
    df['duration'] = df.lpep_dropoff_datetime-df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df=df[((df.duration>=1) & (df.duration<=60))]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical]=df[categorical].astype(str)

    return df

def preprocess(df, dv):
    df['PU_DO']=df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts= df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)

def test_model(X_test, y_test, model_name='nyc-taxi-reg',model_version='latest'):
    
    model_uri = f"models:/{model_name}/{model_version}"
    model=mlflow.pyfunc.load_model(model_uri)
    y_pred = model.predict(X_test)
    return {"rmse":root_mean_squared_error(y_test, y_pred)}

In [27]:
df_train=read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-01.parquet')
df_val=read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-02.parquet')


In [28]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI= "/workspaces/MLOPS/03-Experiment_tracking/mlflow.db"

client=MlflowClient(tracking_uri=f"sqlite:///{MLFLOW_TRACKING_URI}")

run_id = "43f1302b78644865804f2ac69712e673"
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/workspaces/MLOPS/04-Model_registry/preprocessor'

In [29]:
import pickle

with open("preprocessor/preprocessor.b","rb") as f_in:
    dv=pickle.load(f_in)

In [30]:
X_test = preprocess(df_val,dv)

In [31]:
target = "duration"
y_test=df_val[target].values

In [32]:
%time test_model(X_test, y_test)

CPU times: user 1.91 s, sys: 199 ms, total: 2.11 s
Wall time: 4.95 s


{'rmse': 6.394752589852737}